# Chạy QA từ một link artifact Google Drive khác

Notebook này **không chạy lại extraction/submission**. Nó nhận một link folder `run_root` đã được share, tải các artifact cần cho QA, kiểm contract + SHA-256, build BGE-M3 nếu folder share chưa có, rồi chạy grounded QA bằng Qwen3.5-9B 4-bit.

> Link folder không đồng nghĩa artifact đã đủ. Notebook fail-closed nếu thiếu coarse FAISS, frame map, FAISS manifest, text index hoặc `segments_all.jsonl`.

## Cách chạy

1. Mở Colab, chọn GPU T4/L4/A100.
2. Sửa `QA_QUERY` thành câu **chắc chắn có đáp án** trong corpus.
3. Chạy tuần tự từ trên xuống và cấp quyền Google Drive khi Colab hỏi.
4. Lần đầu cần mạng để tải BGE, reranker, SigLIP2 và Qwen.
5. `status=passed` chỉ là smoke contract; chưa phải điểm chất lượng QA.

In [ ]:
# parameters — chỉ sửa giá trị trong cell này
from pathlib import Path

SHARED_ARTIFACT_FOLDER_URL = "https://drive.google.com/drive/folders/13DXIR3E-nPntJYfDkY0-Dpe2vm4dAL-6?usp=drive_link"
GIT_URL = "https://github.com/24122013/AIChallenge26_Multimodal_Agentic_Video_Retrieval_System.git"
GIT_BRANCH = "main"

QA_QUERY = "Trong cảnh đường phố có phương tiện gì?"
TASK_TOP_K = 5
DEVICE = "cuda"
MIN_GPU_MEMORY_MIB = 14000

BGE_M3_MODEL = "BAAI/bge-m3"
BGE_M3_REVISION = "main"  # pin commit hash trước benchmark chính thức
BGE_RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
BGE_RERANKER_REVISION = "main"
BGE_BATCH_SIZE = 8
BGE_RERANKER_ALPHA = 0.5

QA_MODEL = "Qwen/Qwen3.5-9B"
QA_MODEL_REVISION = "c202236235762e1c871ad0ccb60c8ee5ba337b9a"
QA_QUANTIZATION = "4bit"

WORKSPACE = Path("/content/ai_challenge26")
ARTIFACT_ROOT = Path("/content/shared_qa_artifacts")
RUNTIME_ROOT = Path("/content/rebased_qa_artifacts")
MODEL_CACHE_ROOT = Path("/content/drive/MyDrive/AIChallenge26/model_cache")
OUTPUT_ROOT = Path("/content/drive/MyDrive/AIChallenge26/qa_from_shared_link")
QA_OUTPUT = OUTPUT_ROOT / "qa_smoke.json"

# Chỉ tải phần QA cần; bỏ work/dense/embeddings để tiết kiệm thời gian và disk.
ARTIFACT_BRANCHES = {"indexes", "metadata", "keyframes", "run_manifest.json"}


## 1. Mount Drive, kiểm GPU và lấy source mới nhất

In [ ]:
import os
import re
import shutil
import subprocess
import sys

os.chdir("/content")
from google.colab import auth, drive

drive.mount("/content/drive")
auth.authenticate_user()

gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    text=True, capture_output=True, check=False,
)
if gpu_query.returncode != 0 or not gpu_query.stdout.strip():
    raise RuntimeError("Không thấy NVIDIA GPU. Hãy đổi Colab runtime sang GPU.")
gpu_name, gpu_memory = [item.strip() for item in gpu_query.stdout.splitlines()[0].rsplit(",", 1)]
gpu_memory_mib = int(gpu_memory)
print({"gpu": gpu_name, "memory_mib": gpu_memory_mib})
if DEVICE == "cuda" and gpu_memory_mib < MIN_GPU_MEMORY_MIB:
    raise RuntimeError(
        f"GPU chỉ có {gpu_memory_mib} MiB; grounded QA 9B 4-bit cần runtime >= {MIN_GPU_MEMORY_MIB} MiB."
    )

from google.colab import userdata

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Thiếu Colab secret GITHUB_TOKEN hoặc Notebook access chưa bật.")
askpass = Path("/tmp/github_askpass.sh")
askpass.write_text(
    '#!/bin/sh\ncase \"$1\" in\n  *Username*) echo x-access-token ;;\n  *) echo \"$GITHUB_TOKEN\" ;;\nesac\n',
    encoding="utf-8",
)
askpass.chmod(0o700)
clone_environment = os.environ.copy()
clone_environment.update({
    "GITHUB_TOKEN": github_token,
    "GIT_ASKPASS": str(askpass),
    "GIT_TERMINAL_PROMPT": "0",
})
probe = subprocess.run(
    ["git", "ls-remote", "--heads", GIT_URL, GIT_BRANCH],
    text=True, capture_output=True, env=clone_environment, check=False,
)
if probe.returncode != 0 or not probe.stdout.strip():
    askpass.unlink(missing_ok=True)
    raise RuntimeError(
        f"Không đọc được branch {GIT_BRANCH!r} từ GitHub. stderr:\n{probe.stderr[-4000:]}"
    )

clone_errors = []
for attempt in range(1, 3):
    if WORKSPACE.exists():
        shutil.rmtree(WORKSPACE)
    clone = subprocess.run(
        [
            "git", "-c", "http.version=HTTP/1.1", "clone",
            "--depth", "1", "--branch", GIT_BRANCH, GIT_URL, str(WORKSPACE),
        ],
        text=True, capture_output=True, env=clone_environment, check=False,
    )
    if clone.returncode == 0:
        break
    clone_errors.append(f"attempt {attempt}: {clone.stderr[-4000:]}")
else:
    askpass.unlink(missing_ok=True)
    raise RuntimeError("Git clone thất bại sau 2 lần:\n" + "\n---\n".join(clone_errors))
print(clone.stdout)
os.chdir(WORKSPACE)
print("commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
askpass.unlink(missing_ok=True)


## 2. Cài runtime QA

Giữ Torch CUDA có sẵn của Colab; chỉ cài dependency repository. Notebook này không chạy OCR/extraction nên không cài PaddlePaddle.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

runtime_probe = subprocess.run(
    [sys.executable, "-c", "import torch, transformers, bitsandbytes, faiss; print(torch.__version__, torch.cuda.is_available(), transformers.__version__)"],
    text=True, capture_output=True, check=False,
)
print(runtime_probe.stdout)
if runtime_probe.returncode != 0:
    raise RuntimeError("QA runtime probe thất bại:\n" + runtime_probe.stderr[-4000:])


## 3. Tải artifact trực tiếp từ folder ID

Không cần Add shortcut to My Drive. Downloader dùng Google Drive API, phân trang đầy đủ và resume file đã tải đúng size.

In [ ]:
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

folder_match = re.search(r"/folders/([A-Za-z0-9_-]+)", SHARED_ARTIFACT_FOLDER_URL)
if not folder_match:
    raise ValueError("SHARED_ARTIFACT_FOLDER_URL không phải link Google Drive folder hợp lệ")
root_folder_id = folder_match.group(1)
drive_api = build("drive", "v3")
FOLDER_MIME = "application/vnd.google-apps.folder"

def safe_child(parent: Path, name: str) -> Path:
    if not name or name in {".", ".."} or Path(name).name != name:
        raise ValueError(f"Tên file Drive không an toàn: {name!r}")
    child = (parent / name).resolve()
    if parent.resolve() not in child.parents:
        raise ValueError(f"Drive path thoát artifact root: {child}")
    return child

def list_children(folder_id: str):
    page_token = None
    while True:
        response = drive_api.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken,files(id,name,mimeType,size,md5Checksum)",
            pageSize=1000,
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        yield from response.get("files", [])
        page_token = response.get("nextPageToken")
        if not page_token:
            break

def download_file(item: dict, destination: Path):
    expected_size = int(item.get("size") or -1)
    if destination.is_file() and expected_size >= 0 and destination.stat().st_size == expected_size:
        return "cached"
    if item["mimeType"].startswith("application/vnd.google-apps."):
        raise RuntimeError(f"Artifact không được là Google-native file: {item['name']}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_suffix(destination.suffix + ".part")
    request = drive_api.files().get_media(fileId=item["id"], supportsAllDrives=True)
    with partial.open("wb") as stream:
        downloader = MediaIoBaseDownload(stream, request, chunksize=16 * 1024 * 1024)
        done = False
        while not done:
            _, done = downloader.next_chunk(num_retries=5)
    partial.replace(destination)
    if expected_size >= 0 and destination.stat().st_size != expected_size:
        raise IOError(f"Sai size sau download: {destination}")
    return "downloaded"

def download_tree(folder_id: str, destination: Path):
    destination.mkdir(parents=True, exist_ok=True)
    counters = {"downloaded": 0, "cached": 0, "folders": 0}
    for item in list_children(folder_id):
        target = safe_child(destination, item["name"])
        if item["mimeType"] == FOLDER_MIME:
            child = download_tree(item["id"], target)
            counters["folders"] += 1 + child["folders"]
            counters["downloaded"] += child["downloaded"]
            counters["cached"] += child["cached"]
        else:
            counters[download_file(item, target)] += 1
    return counters

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
root_items = list(list_children(root_folder_id))
available = {item["name"] for item in root_items}
missing_branches = ARTIFACT_BRANCHES - available
if missing_branches:
    raise FileNotFoundError(f"Folder share thiếu nhánh bắt buộc: {sorted(missing_branches)}")

summary = {"downloaded": 0, "cached": 0, "folders": 0}
for item in root_items:
    if item["name"] not in ARTIFACT_BRANCHES:
        continue
    target = safe_child(ARTIFACT_ROOT, item["name"])
    if item["mimeType"] == FOLDER_MIME:
        current = download_tree(item["id"], target)
    else:
        state = download_file(item, target)
        current = {"downloaded": int(state == "downloaded"), "cached": int(state == "cached"), "folders": 0}
    for key in summary:
        summary[key] += current[key]
    print(item["name"], current)
print("download summary:", summary)


## 4. Validate artifact contract và lineage

In [ ]:
import hashlib
import json

artifact_tag = "siglip2_so400m_patch16_384"
required_paths = {
    "coarse_index": ARTIFACT_ROOT / "indexes" / f"{artifact_tag}_flat_ip.faiss",
    "frame_map": ARTIFACT_ROOT / "metadata" / f"{artifact_tag}_frame_map.json",
    "faiss_manifest": ARTIFACT_ROOT / "metadata" / f"{artifact_tag}_faiss_manifest.json",
    "text_index": ARTIFACT_ROOT / "indexes" / "retrieval_text_index.json",
    "segments": ARTIFACT_ROOT / "metadata" / "segments_all.jsonl",
    "run_manifest": ARTIFACT_ROOT / "run_manifest.json",
}
missing = [f"{name}: {path}" for name, path in required_paths.items() if not path.is_file()]
if missing:
    raise FileNotFoundError("Folder artifact chưa đủ để chạy QA:\n- " + "\n- ".join(missing))

run_manifest = json.loads(required_paths["run_manifest"].read_text(encoding="utf-8"))

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

hash_failures = []
for name, record in run_manifest.get("artifacts", {}).items():
    relative = record.get("path") if isinstance(record, dict) else None
    expected = record.get("sha256") if isinstance(record, dict) else None
    if not relative or not expected:
        continue
    path = ARTIFACT_ROOT / relative
    if path.is_file() and sha256_file(path) != expected:
        hash_failures.append(name)
if hash_failures:
    raise RuntimeError(f"Artifact SHA-256 không khớp run_manifest: {hash_failures}")

print({
    "run_id": run_manifest.get("run_id"),
    "source_git": run_manifest.get("git", {}),
    "run_status": run_manifest.get("status"),
    "required_artifacts": {name: str(path) for name, path in required_paths.items()},
})

# Giữ nguyên artifact tải về để hash vẫn audit được; tạo bản runtime riêng
# vì frame-map/text-index của máy nguồn có thể chứa absolute image path.
if RUNTIME_ROOT.exists():
    shutil.rmtree(RUNTIME_ROOT)
(RUNTIME_ROOT / "metadata").mkdir(parents=True, exist_ok=True)
(RUNTIME_ROOT / "indexes").mkdir(parents=True, exist_ok=True)

def rebase_image_path(value):
    if not isinstance(value, str) or not value.strip():
        return value
    current = Path(value)
    if current.is_file():
        return str(current)
    parts = [part for part in value.replace("\\", "/").split("/") if part]
    if "keyframes" not in parts:
        return value
    marker = parts.index("keyframes")
    return str(ARTIFACT_ROOT.joinpath(*parts[marker:]))

PATH_FIELDS = {"keyframe_path", "thumbnail_path", "image_path", "frame_path"}

def rebase_record(value):
    if isinstance(value, dict):
        return {
            key: (rebase_image_path(item) if key in PATH_FIELDS else rebase_record(item))
            for key, item in value.items()
        }
    if isinstance(value, list):
        return [rebase_record(item) for item in value]
    return value

runtime_frame_map = RUNTIME_ROOT / "metadata" / required_paths["frame_map"].name
frame_map_payload = json.loads(required_paths["frame_map"].read_text(encoding="utf-8"))
runtime_frame_map.write_text(
    json.dumps(rebase_record(frame_map_payload), ensure_ascii=False, separators=(",", ":")),
    encoding="utf-8",
)

runtime_text_index = RUNTIME_ROOT / "indexes" / required_paths["text_index"].name
text_index_payload = json.loads(required_paths["text_index"].read_text(encoding="utf-8"))
runtime_text_index.write_text(
    json.dumps(rebase_record(text_index_payload), ensure_ascii=False, separators=(",", ":")),
    encoding="utf-8",
)

runtime_segments = RUNTIME_ROOT / "metadata" / required_paths["segments"].name
with required_paths["segments"].open(encoding="utf-8") as source_stream, runtime_segments.open("w", encoding="utf-8") as target_stream:
    for line_number, line in enumerate(source_stream, start=1):
        if not line.strip():
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(f"segments_all.jsonl lỗi dòng {line_number}") from exc
        target_stream.write(json.dumps(rebase_record(record), ensure_ascii=False) + "\n")

runtime_paths = {
    "frame_map": runtime_frame_map,
    "text_index": runtime_text_index,
    "segments": runtime_segments,
}
print("runtime paths rebased:", {key: str(path) for key, path in runtime_paths.items()})


## 5. Build hoặc validate BGE-M3 QA index

Run artifact hiện tại có thể chưa chứa `indexes/bge_m3`. Khi thiếu, notebook build từ canonical `segments_all.jsonl`; không chạy lại video extraction.

In [ ]:
BGE_ROOT = RUNTIME_ROOT / "indexes" / "bge_m3"
BGE_REQUIRED = ("bge_m3_flat_ip.faiss", "bge_m3_frame_map.json", "bge_m3_manifest.json")
bge_missing = [name for name in BGE_REQUIRED if not (BGE_ROOT / name).is_file()]
if bge_missing:
    print("BGE artifact thiếu, bắt đầu build:", bge_missing)
    subprocess.run([
        sys.executable, "-m", "backend.app.services.indexing.build_bge_m3_index",
        "--metadata", str(runtime_paths["segments"]),
        "--output-root", str(BGE_ROOT),
        "--model-name", BGE_M3_MODEL,
        "--model-revision", BGE_M3_REVISION,
        "--batch-size", str(BGE_BATCH_SIZE),
        "--device", DEVICE,
        "--cache-dir", str(MODEL_CACHE_ROOT / "bge_m3"),
        "--canonical-only",
    ], cwd=WORKSPACE, check=True)

subprocess.run([
    sys.executable, "-c",
    "from backend.app.services.retrieval.bge_dense import validate_bge_m3_artifacts; "
    f"print(validate_bge_m3_artifacts(r'{BGE_ROOT}'))",
], cwd=WORKSPACE, check=True)


## 6. Chạy strict grounded-QA smoke

Smoke bắt buộc cache miss, model được invoke, câu trả lời có citation và evidence tồn tại. Vì vậy câu hỏi unanswerable sẽ fail smoke dù abstain có thể hợp lý trong production.

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
qa_env = os.environ.copy()
qa_env.update({
    "PYTHONUTF8": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "HF_HOME": str(MODEL_CACHE_ROOT / "huggingface"),
    "RETRIEVAL_INDEX_PATH": str(required_paths["coarse_index"]),
    "RETRIEVAL_FRAME_MAP_PATH": str(runtime_paths["frame_map"]),
    "RETRIEVAL_MANIFEST_PATH": str(required_paths["faiss_manifest"]),
    "RETRIEVAL_TEXT_INDEX_PATH": str(runtime_paths["text_index"]),
    "RETRIEVAL_MODEL_CACHE_DIR": str(MODEL_CACHE_ROOT / "huggingface"),
    "RETRIEVAL_DEVICE": DEVICE,
    "QA_TYPED_PARSER_ENABLED": "true",
    "QA_ROUTER_ENABLED": "true",
    "QA_EVIDENCE_BUNDLE_ENABLED": "true",
    "QA_BGE_DENSE_ENABLED": "true",
    "QA_BGE_RERANKER_ENABLED": "true",
    "QA_BGE_INDEX_ROOT": str(BGE_ROOT),
    "QA_BGE_MODEL_REVISION": BGE_M3_REVISION,
    "QA_BGE_RERANKER_MODEL": BGE_RERANKER_MODEL,
    "QA_BGE_RERANKER_REVISION": BGE_RERANKER_REVISION,
    "QA_BGE_RERANKER_ALPHA": str(BGE_RERANKER_ALPHA),
    "QA_BGE_BATCH_SIZE": str(BGE_BATCH_SIZE),
    "QA_BGE_DEVICE": DEVICE,
    "QA_BGE_MODEL_CACHE_DIR": str(MODEL_CACHE_ROOT / "bge_m3"),
    "QA_ANSWER_MODE": "required",
    "QA_ANSWER_MODEL": QA_MODEL,
    "QA_ANSWER_MODEL_REVISION": QA_MODEL_REVISION,
    "QA_ANSWER_DEVICE": DEVICE,
    "QA_ANSWER_QUANTIZATION": QA_QUANTIZATION,
    "QA_ANSWER_MODEL_CACHE_DIR": str(MODEL_CACHE_ROOT / "qa_answer"),
    "QA_ANSWER_CACHE_DIR": str(Path("/content/qa_answer_cache_miss")),
    "QA_EXPERIMENT_ID": f"{run_manifest.get('run_id', 'shared')}-qa-link-smoke",
})

qa_command = [
    sys.executable, "-m", "backend.app.services.retrieval.run_task_smoke",
    "--task", "qa",
    "--top-k", str(TASK_TOP_K),
    "--qa-query", QA_QUERY,
    "--output", str(QA_OUTPUT),
]
print("QA query:", QA_QUERY)
subprocess.run(qa_command, cwd=WORKSPACE, env=qa_env, check=True)
qa_result = json.loads(QA_OUTPUT.read_text(encoding="utf-8"))
print(json.dumps(qa_result, ensure_ascii=False, indent=2)[:16000])


## 7. Mở evidence để kiểm bằng mắt

In [ ]:
from IPython.display import display
from PIL import Image

qa_payload = qa_result.get("results", [{}])[0]
print("answer:", qa_payload.get("answer"))
for item in qa_payload.get("evidence", [])[:TASK_TOP_K]:
    image_path = Path(str(item.get("image_path") or ""))
    if not image_path.is_absolute():
        image_path = ARTIFACT_ROOT / image_path
    print({key: item.get(key) for key in ("evidence_id", "video_id", "frame_id", "timestamp", "caption")})
    if image_path.is_file():
        display(Image.open(image_path).convert("RGB"))
    else:
        print("Không tìm thấy ảnh evidence:", image_path)


## Output bàn giao

- `/content/drive/MyDrive/AIChallenge26/qa_from_shared_link/qa_smoke.json`
- `status=passed`: pipeline QA + model invocation + citation contract chạy được.
- Muốn kết luận chất lượng phải dùng dev labels/gold evidence; một smoke query không đủ.